## **Lab Assignment 3: Text Classification with Deep Learning**
George Danut ASU ID: [redacted]

**Objective:** In this lab assignment, you will use some deep learning models to quantify sentiments on the Yelp Review data and compare their performance.

## Import all the necessary libraries and restaurant review data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt

# Train/test splitting
from sklearn.model_selection import train_test_split

# Model evaluation
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.models import Sequential

# Text preprocessing
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Neural network layers
from tensorflow.keras.layers import (
    Embedding,
    SimpleRNN,
    LSTM,
    GRU,
    Dense,
    Dropout,
    Bidirectional,
    GlobalAveragePooling1D
)

# Training callbacks
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
# Import the restaurant review dataset
restaurant_reviews = pd.read_csv('data/restaurant_reviews_az.csv')

# Check the dimensions of the dataset
print("Dataset shape:", restaurant_reviews.shape)

# Display column names
print("\nColumns:")
print(restaurant_reviews.columns.tolist())

# Display basic dataset information
print("\nDataset Information:")
restaurant_reviews.info()

## Remove 3-star reviews from the input data and create a new column - Sentiment for the remaining reviews. For reviews with 1 or 2 star ratings, set the value in the Sentiment column to 0. For reviews with 4 or 5-star ratings, set the value in the sentiment column to 1.

In [ ]:
# Remove all 3-star reviews from the dataset
restaurant_reviews = restaurant_reviews[
    restaurant_reviews['stars'] != 3
].copy()

# Create the Sentiment column
# 1 or 2 stars = 0 (Negative)
# 4 or 5 stars = 1 (Positive)
restaurant_reviews['Sentiment'] = (
    restaurant_reviews['stars'] >= 4
).astype(int)

# Display the first few rows
restaurant_reviews.head()

##Conduct necessary data processing. Prepare the training and test sets on review data for machine learning classifications. 20% of the data is for testing, and 80% of the data is for training.

In [ ]:
## Text Preprocessing
import spacy
from tqdm.auto import tqdm
# Load spaCy English model
nlp = spacy.load(
    "en_core_web_sm",
    disable=["ner", "parser", "textcat"]
)

# Function to normalize a single review
def normalize_single_review(doc, lowercase=True, remove_stopwords=True):

    lemmatized = []

    for token in doc:

        # Convert each word to its lemma/base form
        lemma = token.lemma_.lower() if lowercase else token.lemma_

        # Remove stopwords such as "the", "is", "at", etc.
        if remove_stopwords and token.is_stop:
            continue

        # Keep alphabetic words
        if token.is_alpha:
            lemmatized.append(lemma)

    return " ".join(lemmatized)


# Process all restaurant reviews using spaCy
processed_reviews = []

for doc in tqdm(
    nlp.pipe(restaurant_reviews['text'], batch_size=500),
    total=len(restaurant_reviews),
    desc="Processing Reviews"
):
    processed_reviews.append(normalize_single_review(doc))


# Add processed text to the dataframe
restaurant_reviews['processed_text'] = processed_reviews

In [ ]:
## Train Test Split

# Split data into training and testing sets
# 80% Training / 20% Testing

train_sentences, test_sentences, train_labels, test_labels = train_test_split(
    restaurant_reviews['processed_text'],  # Input features
    restaurant_reviews['Sentiment'],       # Target variable
    test_size=0.20,                        # 20% testing
    random_state=5,                        # Reproducibility
    stratify=restaurant_reviews['Sentiment']  # Maintain class distribution
)

# Display the sizes of each dataset
print("Training samples:", len(train_sentences))
print("Testing samples:", len(test_sentences))

print("\nTraining label distribution:")
print(train_labels.value_counts())

print("\nTesting label distribution:")
print(test_labels.value_counts())

In [ ]:
## Text Sequence Parameters
VOCAB_SIZE = 20000
MAX_LENGTH = 100
PADDING_TYPE = 'post'
TRUNC_TYPE = 'post'
OOV_TOKEN = "<OOV>"

In [ ]:
# Create tokenizer
tokenizer = tf.keras.preprocessing.text.Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token=OOV_TOKEN
)

# Fit tokenizer ONLY on training reviews
tokenizer.fit_on_texts(train_sentences)

# Convert training and test reviews into integer sequences
train_sequences = tokenizer.texts_to_sequences(train_sentences)
test_sequences = tokenizer.texts_to_sequences(test_sentences)

In [ ]:
## Pad Sequences

train_padded = pad_sequences(
    train_sequences,
    maxlen=MAX_LENGTH,
    padding=PADDING_TYPE,
    truncating=TRUNC_TYPE
)

test_padded = pad_sequences(
    test_sequences,
    maxlen=MAX_LENGTH,
    padding=PADDING_TYPE,
    truncating=TRUNC_TYPE
)

# Convert labels to NumPy arrays
train_labels = np.array(train_labels)
test_labels = np.array(test_labels)

In [ ]:
# Check final dimensions
print("Training data shape:", train_padded.shape)
print("Training labels shape:", train_labels.shape)

print("\nTesting data shape:", test_padded.shape)
print("Testing labels shape:", test_labels.shape)

## Download the pre-trained GloVe word embeddings (glove.6B.100d.txt) and prepare the embedding matrix for this review dataset (refer to Module 5's sample code in python tutorial session).

In [ ]:
# Download the GloVe 6B pretrained word embeddings
!wget -q https://nlp.stanford.edu/data/glove.6B.zip

# Unzip the downloaded file
!unzip -q glove.6B.zip

print("GloVe embeddings downloaded and extracted.")

In [ ]:
## Load Glove Embeddings

# Each GloVe word is represented by 100 values
EMBEDDING_DIM = 100

# Dictionary to store:
# word -> 100-dimensional GloVe vector
embeddings_index = {}

# Open the GloVe file
with open("glove.6B.100d.txt", encoding="utf-8") as f:

    for line in f:

        # Split each line into the word and its vector values
        values = line.split()

        # First value is the actual word
        word = values[0]

        # Remaining 100 values are the word embedding
        vector = np.asarray(values[1:], dtype="float32")

        # Store word and vector in dictionary
        embeddings_index[word] = vector


print("Number of GloVe word vectors loaded:",
      len(embeddings_index))

In [ ]:
## Create Matrix

# Determine how many words from our tokenizer will actually be used
num_words = min(VOCAB_SIZE, len(tokenizer.word_index) + 1)

# Create an empty embedding matrix
# Rows = words in our vocabulary
# Columns = 100 GloVe dimensions
embedding_matrix = np.zeros(
    (num_words, EMBEDDING_DIM)
)

# Match each word from our restaurant review vocabulary
# with its corresponding GloVe vector
for word, index in tokenizer.word_index.items():

    # Stop if the word index is outside our vocabulary limit
    if index >= VOCAB_SIZE:
        continue

    # Find the word in the pretrained GloVe dictionary
    embedding_vector = embeddings_index.get(word)

    # If the word exists in GloVe, place its vector
    # into the corresponding row of the embedding matrix
    if embedding_vector is not None:
        embedding_matrix[index] = embedding_vector

In [ ]:
# Count words successfully matched to GloVe
matched_words = 0
unmatched_words = 0

for word, index in tokenizer.word_index.items():

    if index >= VOCAB_SIZE:
        continue

    if word in embeddings_index:
        matched_words += 1
    else:
        unmatched_words += 1


print("Words found in GloVe:", matched_words)
print("Words not found in GloVe:", unmatched_words)

coverage = matched_words / (matched_words + unmatched_words) * 100

print(f"GloVe vocabulary coverage: {coverage:.2f}%")

## Build a GRU model with pre-trained GloVe embedding and show model performance

In [ ]:
gru_model = Sequential([

    # Explicitly define input shape
    tf.keras.Input(shape=(MAX_LENGTH,)),

    # Pre-trained GloVe embedding layer
    Embedding(
        input_dim=num_words,
        output_dim=EMBEDDING_DIM,
        weights=[embedding_matrix],
        trainable=False
    ),

    # GRU layer
    GRU(64),

    # Binary classification output
    Dense(1, activation='sigmoid')
])

# Compile model
gru_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Display architecture
gru_model.summary()

In [ ]:
## Early Stopping
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [ ]:
## Train Gru Model

history_gru = gru_model.fit(
    train_padded,
    train_labels,
    epochs=20,
    batch_size=64,
    validation_split=0.20,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
## Evaluate Model on test data
test_loss, test_accuracy = gru_model.evaluate(
    test_padded,
    test_labels,
    verbose=0
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

In [ ]:
# Generate predicted probabilities
y_pred_prob = gru_model.predict(test_padded)

# Convert probabilities to binary classifications
# Probability >= 0.50 = Positive (1)
# Probability < 0.50 = Negative (0)
y_pred = (y_pred_prob >= 0.5).astype(int).flatten()

# Display classification report
print(classification_report(
    test_labels,
    y_pred,
    target_names=['Negative', 'Positive']
))

In [ ]:
## Confusion Matrix

cm = confusion_matrix(test_labels, y_pred)

print("Confusion Matrix:")
print(cm)

# Display confusion matrix
plt.figure(figsize=(6, 5))

plt.imshow(cm)

plt.title("GRU Model Confusion Matrix")
plt.xlabel("Predicted Sentiment")
plt.ylabel("Actual Sentiment")

plt.xticks([0, 1], ["Negative", "Positive"])
plt.yticks([0, 1], ["Negative", "Positive"])

# Add values inside matrix
for i in range(2):
    for j in range(2):
        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

plt.colorbar()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history_gru.history['accuracy'],
    label='Training Accuracy'
)

plt.plot(
    history_gru.history['val_accuracy'],
    label='Validation Accuracy'
)

plt.title('GRU Model: Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history_gru.history['loss'],
    label='Training Loss'
)

plt.plot(
    history_gru.history['val_loss'],
    label='Validation Loss'
)

plt.title('GRU Model: Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

## Build an LSTM model with pre-trained GloVe embedding and show model performance

In [ ]:
lstm_model = Sequential([

    # Explicitly define input shape
    tf.keras.Input(shape=(MAX_LENGTH,)),

    # Pre-trained GloVe embedding layer
    Embedding(
        input_dim=num_words,
        output_dim=EMBEDDING_DIM,
        weights=[embedding_matrix],
        trainable=False
    ),

    # LSTM layer
    LSTM(64),

    # Binary sentiment classification
    Dense(1, activation='sigmoid')
])

# Compile the model
lstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Display model architecture
lstm_model.summary()

In [ ]:
# Early stopping
early_stopping_lstm = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)


# Train

history_lstm = lstm_model.fit(
    train_padded,
    train_labels,
    epochs=20,
    batch_size=64,
    validation_split=0.20,
    callbacks=[early_stopping_lstm],
    verbose=1
)

In [ ]:
## Evaluate
lstm_test_loss, lstm_test_accuracy = lstm_model.evaluate(
    test_padded,
    test_labels,
    verbose=0
)

print(f"LSTM Test Loss: {lstm_test_loss:.4f}")
print(f"LSTM Test Accuracy: {lstm_test_accuracy:.4f}")
print(f"LSTM Test Accuracy: {lstm_test_accuracy * 100:.2f}%")

In [ ]:
# Generate predicted probabilities
lstm_pred_prob = lstm_model.predict(test_padded)

# Convert probabilities into 0/1 predictions
lstm_pred = (lstm_pred_prob >= 0.5).astype(int).flatten()

# Classification report
print("LSTM Classification Report:\n")

print(classification_report(
    test_labels,
    lstm_pred,
    target_names=['Negative', 'Positive']
))

In [ ]:
lstm_cm = confusion_matrix(
    test_labels,
    lstm_pred
)

print("LSTM Confusion Matrix:")
print(lstm_cm)

plt.figure(figsize=(6, 5))

plt.imshow(lstm_cm)

plt.title("LSTM Model Confusion Matrix")
plt.xlabel("Predicted Sentiment")
plt.ylabel("Actual Sentiment")

plt.xticks([0, 1], ["Negative", "Positive"])
plt.yticks([0, 1], ["Negative", "Positive"])

# Add counts inside each cell
for i in range(2):
    for j in range(2):
        plt.text(
            j,
            i,
            lstm_cm[i, j],
            ha="center",
            va="center"
        )

plt.colorbar()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history_lstm.history['accuracy'],
    label='Training Accuracy'
)

plt.plot(
    history_lstm.history['val_accuracy'],
    label='Validation Accuracy'
)

plt.title('LSTM Model: Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history_lstm.history['loss'],
    label='Training Loss'
)

plt.plot(
    history_lstm.history['val_loss'],
    label='Validation Loss'
)

plt.title('LSTM Model: Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

##Build a GRU model with trainable GloVe embeddings and show model performance

In [ ]:
gru_trainable_model = Sequential([

    # Explicitly define input shape
    tf.keras.Input(shape=(MAX_LENGTH,)),

    # Pre-trained GloVe embedding layer
    # trainable=True allows GloVe embeddings to be fine-tuned
    Embedding(
        input_dim=num_words,
        output_dim=EMBEDDING_DIM,
        weights=[embedding_matrix],
        trainable=True
    ),

    # GRU layer
    GRU(64),

    # Binary sentiment classification output
    Dense(1, activation='sigmoid')
])

# Compile model
gru_trainable_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Display model architecture
gru_trainable_model.summary()

In [ ]:
early_stopping_gru_trainable = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history_gru_trainable = gru_trainable_model.fit(
    train_padded,
    train_labels,
    epochs=20,
    batch_size=64,
    validation_split=0.20,
    callbacks=[early_stopping_gru_trainable],
    verbose=1
)

In [ ]:
# Predicted probabilities
gru_trainable_pred_prob = gru_trainable_model.predict(
    test_padded
)

# Convert probabilities to binary predictions
gru_trainable_pred = (
    gru_trainable_pred_prob >= 0.5
).astype(int).flatten()


# Classification report
print("GRU with Trainable GloVe Classification Report:\n")

print(classification_report(
    test_labels,
    gru_trainable_pred,
    target_names=['Negative', 'Positive']
))

In [ ]:
gru_trainable_cm = confusion_matrix(
    test_labels,
    gru_trainable_pred
)

print("GRU with Trainable GloVe Confusion Matrix:")
print(gru_trainable_cm)


plt.figure(figsize=(6, 5))

plt.imshow(gru_trainable_cm)

plt.title("GRU with Trainable GloVe Confusion Matrix")
plt.xlabel("Predicted Sentiment")
plt.ylabel("Actual Sentiment")

plt.xticks([0, 1], ["Negative", "Positive"])
plt.yticks([0, 1], ["Negative", "Positive"])

# Add values to confusion matrix
for i in range(2):
    for j in range(2):
        plt.text(
            j,
            i,
            gru_trainable_cm[i, j],
            ha="center",
            va="center"
        )

plt.colorbar()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history_gru_trainable.history['accuracy'],
    label='Training Accuracy'
)

plt.plot(
    history_gru_trainable.history['val_accuracy'],
    label='Validation Accuracy'
)

plt.title('GRU Trainable GloVe: Training vs Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history_gru_trainable.history['loss'],
    label='Training Loss'
)

plt.plot(
    history_gru_trainable.history['val_loss'],
    label='Validation Loss'
)

plt.title('GRU Trainable GloVe: Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

## Build an LSTM model with trainable GloVe embeddings and show model performance


In [ ]:
lstm_trainable_model = Sequential([

    # Explicitly define input shape
    tf.keras.Input(shape=(MAX_LENGTH,)),

    # Pre-trained GloVe embedding layer
    # trainable=True allows embeddings to be fine-tuned
    Embedding(
        input_dim=num_words,
        output_dim=EMBEDDING_DIM,
        weights=[embedding_matrix],
        trainable=True
    ),

    # LSTM layer
    LSTM(64),

    # Binary sentiment classification output
    Dense(1, activation='sigmoid')
])

# Compile the model
lstm_trainable_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Display model architecture
lstm_trainable_model.summary()

In [ ]:
early_stopping_lstm_trainable = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history_lstm_trainable = lstm_trainable_model.fit(
    train_padded,
    train_labels,
    epochs=20,
    batch_size=64,
    validation_split=0.20,
    callbacks=[early_stopping_lstm_trainable],
    verbose=1
)

In [ ]:
lstm_trainable_test_loss, lstm_trainable_test_accuracy = (
    lstm_trainable_model.evaluate(
        test_padded,
        test_labels,
        verbose=0
    )
)

print(f"LSTM Trainable GloVe Test Loss: {lstm_trainable_test_loss:.4f}")
print(f"LSTM Trainable GloVe Test Accuracy: {lstm_trainable_test_accuracy:.4f}")
print(f"LSTM Trainable GloVe Test Accuracy: {lstm_trainable_test_accuracy * 100:.2f}%")

In [ ]:
# Generate predicted probabilities
lstm_trainable_pred_prob = lstm_trainable_model.predict(
    test_padded
)

# Convert probabilities to binary predictions
# < 0.50 = Negative
# >= 0.50 = Positive
lstm_trainable_pred = (
    lstm_trainable_pred_prob >= 0.5
).astype(int).flatten()


# Classification report
print("LSTM with Trainable GloVe Classification Report:\n")

print(classification_report(
    test_labels,
    lstm_trainable_pred,
    target_names=['Negative', 'Positive']
))

In [ ]:
lstm_trainable_cm = confusion_matrix(
    test_labels,
    lstm_trainable_pred
)

print("LSTM with Trainable GloVe Confusion Matrix:")
print(lstm_trainable_cm)


plt.figure(figsize=(6, 5))

plt.imshow(lstm_trainable_cm)

plt.title("LSTM with Trainable GloVe Confusion Matrix")
plt.xlabel("Predicted Sentiment")
plt.ylabel("Actual Sentiment")

plt.xticks([0, 1], ["Negative", "Positive"])
plt.yticks([0, 1], ["Negative", "Positive"])

# Add values inside confusion matrix
for i in range(2):
    for j in range(2):
        plt.text(
            j,
            i,
            lstm_trainable_cm[i, j],
            ha="center",
            va="center"
        )

plt.colorbar()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history_lstm_trainable.history['accuracy'],
    label='Training Accuracy'
)

plt.plot(
    history_lstm_trainable.history['val_accuracy'],
    label='Validation Accuracy'
)

plt.title("LSTM Trainable GloVe: Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history_lstm_trainable.history['loss'],
    label='Training Loss'
)

plt.plot(
    history_lstm_trainable.history['val_loss'],
    label='Validation Loss'
)

plt.title("LSTM Trainable GloVe: Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()

## Use the best Model from Lab 2 (SVM with TF-IDF) and evaluate performance

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# Create TF-IDF vectorizer
# Use the same 1000-feature limit from the previous notebook
tfidf_vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 1),
    max_features=1000
)

# Fit TF-IDF ONLY on training data
X_train_tfidf = tfidf_vectorizer.fit_transform(train_sentences)

# Transform test data using the SAME fitted vectorizer
X_test_tfidf = tfidf_vectorizer.transform(test_sentences)

# Check dimensions
print("Training TF-IDF Shape:", X_train_tfidf.shape)
print("Testing TF-IDF Shape:", X_test_tfidf.shape)

In [ ]:
svm_tfidf_model = SVC()

# Train SVM using TF-IDF features
svm_tfidf_model.fit(
    X_train_tfidf,
    train_labels
)

print("SVM training complete.")

In [ ]:
svm_tfidf_pred = svm_tfidf_model.predict(
    X_test_tfidf
)

In [ ]:
print("SVM with TF-IDF - Testing Performance:\n")

print(classification_report(
    test_labels,
    svm_tfidf_pred,
    target_names=['Negative', 'Positive']
))

In [ ]:
svm_tfidf_train_pred = svm_tfidf_model.predict(
    X_train_tfidf
)

print("SVM with TF-IDF - Training Performance:\n")

print(classification_report(
    train_labels,
    svm_tfidf_train_pred,
    target_names=['Negative', 'Positive']
))

print("\nSVM with TF-IDF - Testing Performance:\n")

print(classification_report(
    test_labels,
    svm_tfidf_pred,
    target_names=['Negative', 'Positive']
))

In [ ]:
svm_tfidf_cm = confusion_matrix(
    test_labels,
    svm_tfidf_pred
)

print("SVM with TF-IDF Confusion Matrix:")
print(svm_tfidf_cm)


plt.figure(figsize=(6, 5))

plt.imshow(svm_tfidf_cm)

plt.title("SVM with TF-IDF Confusion Matrix")
plt.xlabel("Predicted Sentiment")
plt.ylabel("Actual Sentiment")

plt.xticks([0, 1], ["Negative", "Positive"])
plt.yticks([0, 1], ["Negative", "Positive"])

for i in range(2):
    for j in range(2):
        plt.text(
            j,
            i,
            svm_tfidf_cm[i, j],
            ha="center",
            va="center"
        )

plt.colorbar()
plt.show()

In [ ]:
## Final Model Performance Comparison
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Store model names and their predictions
models = {
    'GRU - Frozen GloVe': y_pred,
    'LSTM - Frozen GloVe': lstm_pred,
    'GRU - Trainable GloVe': gru_trainable_pred,
    'LSTM - Trainable GloVe': lstm_trainable_pred,
    'SVM - TF-IDF': svm_tfidf_pred
}

# Empty list to store results
results = []

# Calculate performance metrics for each model
for model_name, predictions in models.items():

    accuracy = accuracy_score(test_labels, predictions)
    precision = precision_score(test_labels, predictions)
    recall = recall_score(test_labels, predictions)
    f1 = f1_score(test_labels, predictions)

    results.append({
        'Model': model_name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1
    })

# Convert results into a DataFrame
model_performance = pd.DataFrame(results)

# Sort models from highest to lowest accuracy
model_performance = model_performance.sort_values(
    by='Accuracy',
    ascending=False
).reset_index(drop=True)

# Round values for easier interpretation
model_performance[
    ['Accuracy', 'Precision', 'Recall', 'F1 Score']
] = model_performance[
    ['Accuracy', 'Precision', 'Recall', 'F1 Score']
].round(4)

# Display final comparison
model_performance

Overall, all five models performed very well on the restaurant review sentiment classification task, with test accuracies ranging from 93.91% to 94.52%. The GRU with frozen GloVe embeddings and the LSTM with trainable GloVe embeddings achieved the highest accuracy at 94.52%, although the differences among the models were relatively small. Interestingly, making the GloVe embeddings trainable did not consistently improve performance. The trainable LSTM improved over the frozen LSTM from 93.91% to 94.52%, while the trainable GRU slightly underperformed the frozen GRU, with accuracies of 94.35% and 94.52%, respectively. This suggests that fine-tuning pretrained embeddings can be beneficial, but its effect depends on the recurrent architecture. The traditional SVM with TF-IDF was also highly competitive, achieving 94.51% accuracy and the highest F1-score of 0.9621, despite not using sequential modeling or pretrained word embeddings. The trainable GRU produced the highest recall at 0.9706, while the trainable LSTM produced the highest precision at 0.9665, illustrating small trade-offs in the types of classification errors made by each model. Overall, the narrow performance differences indicate that increased model complexity did not produce a substantial improvement over the simpler SVM with TF-IDF for this dataset.


 ## Acknowledge if you have used any GenAI tools in this assignment and anyone you have worked together with on this assignment.

I worked on this assigment by myself. LLM was used to debug code and verify validity of outputs.

In [ ]:
# HTML Rendering
!pip install jupyter
!pip install nbconvert
!jupyter nbconvert "data/LA3_Danut_George.ipynb" --to html